In [2]:
import os
import re
from pathlib import Path
from io import StringIO

import anndata
import numpy as np
import pandas as pd
import seaborn as sns
import tifffile

import scanpy as sc
import matplotlib.pyplot as plt

In [3]:
meta = pd.read_csv(StringIO("""Case	Library ID	Section_ID_clean	Section ID	Block ID	Project	Cancer type	Tissue site	Primary/met	Output_dir	spaceranger_out_folder	HE	LoupeBrowser_8um	BoxFolder	annotation_filepath
HT891Z1	HT891Z1-S2H3Fp1U2Zc1_1Bh1_1	HT891Z1-S2H3Fp1U2	HT891Z1-S2H3Fp1U2	HT891Z1-S2H3Fp1	HTAN	PCa	Prostate	Primary	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT891Z1-S2H3Fp1/	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT891Z1-S2H3Fp1//HT891Z1-S2H3Fp1U2/outs	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/Image/20240715_HTAN_prostate serial/HT891Z1-S2H3Fp1U2_Scan2.qptiff	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT891Z1-S2H3Fp1//HT891Z1-S2H3Fp1U2/outs/cloupe_008um.cloupe	https://wustl.box.com/s/mdwpqu6hhrl5w8jd7ho5f6v18zr6u7zc	
HT891Z1	HT891Z1-S2H3Fp1U33Zc1_1Bh1_1	HT891Z1-S2H3Fp1U33	HT891Z1-S2H3Fp1U33	HT891Z1-S2H3Fp1	HTAN	PCa	Prostate	Primary	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT891Z1-S2H3Fp1/	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT891Z1-S2H3Fp1//HT891Z1-S2H3Fp1U33/outs	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/Image/20240715_HTAN_prostate serial/HT891Z1-S2H3Fp1U33_Scan1.qptiff	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT891Z1-S2H3Fp1//HT891Z1-S2H3Fp1U33/outs/cloupe_008um.cloupe	https://wustl.box.com/s/bkxkgxxc6074a4qp2okcs40w6auvl3ca	
HT935Z1	HT935Z1-S1H1Fp1U1Zc1_1Bh1_1	HT935Z1-S1H1Fp1U1	HT935Z1-S1H1Fp1U1	HT935Z1-S1H1Fp1	HTAN	PCa	Prostate	Primary	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT935Z1-S1H1Fp1/	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT935Z1-S1H1Fp1//HT935Z1-S1H1Fp1U1/outs	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/Image/20241125_PCa & PKD//20241125-HT935Z1-S1H1Fp1/Scan2/20241125-HT935Z1-S1H1Fp1_Scan2.qptiff	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT935Z1-S1H1Fp1//HT935Z1-S1H1Fp1U1/outs/cloupe_008um.cloupe	https://wustl.box.com/s/idtj7p4hpoouccgc4y0y9s56yx4ixtvq	
HT704B1	HT704B1-S1H3Fp1U2Zc1_1Bh1_1	HT704B1-S1H3Fp1U2	HT704B1-S1H3Fp1U2	HT704B1-S1H3Fp1	HTAN	BRCA	Breast	Primary	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/BRCA/HT704B1-S1H3Fp1/	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/BRCA/HT704B1-S1H3Fp1//HT704B1-S1H3Fp1U2/outs	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/Image/20240726_HTAN_Breast serial/HT704B1-S1H3Fp1U2_Scan3.qptiff	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/BRCA/HT704B1-S1H3Fp1//HT704B1-S1H3Fp1U2/outs/cloupe_008um.cloupe	https://wustl.box.com/s/gqowuinxstoune2j8olhpyuparmo5mew	
HT704B1	HT704B1-S1H3Fp1U51Zc1_1Bh1_1	HT704B1-S1H3Fp1U51	HT704B1-S1H3Fp1U51	HT704B1-S1H3Fp1	HTAN	BRCA	Breast	Primary	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/BRCA/HT704B1-S1H3Fp1/	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/BRCA/HT704B1-S1H3Fp1//HT704B1-S1H3Fp1U51/outs	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/Image/20240726_HTAN_Breast serial/HT704B1-S1H3Fp1U51_Scan1.qptiff	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/BRCA/HT704B1-S1H3Fp1//HT704B1-S1H3Fp1U51/outs/cloupe_008um.cloupe	https://wustl.box.com/s/r5o82pzd4nn4myjuwai4xciuykhvuu57	
HT935Z1	HT935Z1-S1H1Fp1U2Zc1_1Bh1_1	HT935Z1-S1H1Fp1U2	HT935Z1-S1H1Fp1U2	HT935Z1-S1H1Fp1	HTAN	PCa	Prostate	Primary	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT935Z1-S1H1Fp1/	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT935Z1-S1H1Fp1//HT935Z1-S1H1Fp1U2/outs	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/Image/20250129_HTAN_PCa & BRCA//HT935Z1_S1H1Fp1_Scan2.qptiff	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/PCa/HT935Z1-S1H1Fp1//HT935Z1-S1H1Fp1U2/outs/cloupe_008um.cloupe	https://wustl.box.com/s/p53tzajqdfzkrrzzqex1h77e0ppcfv4e	
HT727B1	HT727B1-S1H4Fp1U1Zc1_1Bh1_1	HT727B1-S1H4Fp1U1	HT727B1-S1H4Fp1U1	HT727B1-S1H4Fp1	HTAN	BRCA	Breast	Primary	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/BRCA/HT727B1-S1H4Fp1/	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/BRCA/HT727B1-S1H4Fp1//HT727B1-S1H4Fp1U1/outs	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/Image/20250129_HTAN_PCa & BRCA//20250129_HT727B1-S1H4Fp1_Scan2.qptiff	/diskmnt/primary/Spatial_Transcriptomics/VisiumHD_run/FFPE/SpacerangerOut/HTAN/BRCA/HT727B1-S1H4Fp1//HT727B1-S1H4Fp1U1/outs/cloupe_008um.cloupe	https://wustl.box.com/s/d431s9del3851eyzkvwzq4hyh7gbduzm	"""), sep='\t')

meta

,Case,Library ID,Section_ID_clean,Section ID,Block ID,Project,Cancer type,Tissue site,Primary/met,Output_dir,spaceranger_out_folder,HE,LoupeBrowser_8um,BoxFolder,annotation_filepath
0,HT891Z1,HT891Z1-S2H3Fp1U2Zc1_1Bh1_1,HT891Z1-S2H3Fp1U2,HT891Z1-S2H3Fp1U2,HT891Z1-S2H3Fp1,HTAN,PCa,Prostate,Primary,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,https://wustl.box.com/s/mdwpqu6hhrl5w8jd7ho5f6...,NaN
1,HT891Z1,HT891Z1-S2H3Fp1U33Zc1_1Bh1_1,HT891Z1-S2H3Fp1U33,HT891Z1-S2H3Fp1U33,HT891Z1-S2H3Fp1,HTAN,PCa,Prostate,Primary,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,https://wustl.box.com/s/bkxkgxxc6074a4qp2okcs4...,NaN
2,HT935Z1,HT935Z1-S1H1Fp1U1Zc1_1Bh1_1,HT935Z1-S1H1Fp1U1,HT935Z1-S1H1Fp1U1,HT935Z1-S1H1Fp1,HTAN,PCa,Prostate,Primary,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,https://wustl.box.com/s/idtj7p4hpoouccgc4y0y9s...,NaN
3,HT704B1,HT704B1-S1H3Fp1U2Zc1_1Bh1_1,HT704B1-S1H3Fp1U2,HT704B1-S1H3Fp1U2,HT704B1-S1H3Fp1,HTAN,BRCA,Breast,Primary,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,https://wustl.box.com/s/gqowuinxstoune2j8olhpy...,NaN
4,HT704B1,HT704B1-S1H3Fp1U51Zc1_1Bh1_1,HT704B1-S1H3Fp1U51,HT704B1-S1H3Fp1U51,HT704B1-S1H3Fp1,HTAN,BRCA,Breast,Primary,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,https://wustl.box.com/s/r5o82pzd4nn4myjuwai4xc...,NaN
5,HT935Z1,HT935Z1-S1H1Fp1U2Zc1_1Bh1_1,HT935Z1-S1H1Fp1U2,HT935Z1-S1H1Fp1U2,HT935Z1-S1H1Fp1,HTAN,PCa,Prostate,Primary,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,https://wustl.box.com/s/p53tzajqdfzkrrzzqex1h7...,NaN
6,HT727B1,HT727B1-S1H4Fp1U1Zc1_1Bh1_1,HT727B1-S1H4Fp1U1,HT727B1-S1H4Fp1U1,HT727B1-S1H4Fp1,HTAN,BRCA,Breast,Primary,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,/diskmnt/primary/Spatial_Transcriptomics/Visiu...,https://wustl.box.com/s/d431s9del3851eyzkvwzq4...,NaN


In [22]:
vishd_dir = Path('../../visiumhd/')
output_dir = Path('../data/spatial/inputs/visiumhd')

In [24]:
for i, row in meta.iterrows():
    sid = row['Section_ID_clean']
    print(sid)
    
    outs = Path(re.sub(r'^(.*)/HTAN/(PCa|BRCA)/(.*)$', str(vishd_dir) + r'/\3', row['spaceranger_out_folder']))
    
    adata = sc.read_10x_h5(outs / 'binned_outputs/square_002um/filtered_feature_bc_matrix.h5')
    
    positions = pd.read_parquet(
        outs / 'binned_outputs/square_002um/spatial/tissue_positions.parquet',
        engine="pyarrow"
    )
    positions = positions[positions['in_tissue'] == 1]
    positions = positions.set_index('barcode')
    positions = positions.loc[adata.obs.index]

    adata.obsm['spatial'] = positions[['array_col', 'array_row']].values
    adata.obs['x_location'] = positions['pxl_row_in_fullres']
    adata.obs['y_location'] = positions['pxl_col_in_fullres']
    
    adata.write_h5ad(output_dir / f'{sid}_counts.h5ad')

HT891Z1-S2H3Fp1U2


/miniconda/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/miniconda/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


HT891Z1-S2H3Fp1U33


/miniconda/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/miniconda/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


HT935Z1-S1H1Fp1U1


...

/miniconda/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/miniconda/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


HT935Z1-S1H1Fp1U2


/miniconda/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/miniconda/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


HT727B1-S1H4Fp1U1


/miniconda/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/miniconda/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
